# Lecture: Clinical Research Informatics - Sommer Semester 2026
Exercise Sheet: {05}
Student Name: {David_Jansen}
Matriculation Number: {657442}
   
Additional information:   
used additional packages: {pathlib}
dataset used: {Coding Data 02 - standard}

In [ ]:
# Please add this header to all your submissions and adapt it to your information and submission.
# The following structure is a recommendation, but not mandatory to use

# Functions

In [ ]:

import pandas as pd
from pathlib import Path




# Section 1

1.1 ...

In [ ]:
# Lade Datapaths
ALLERGIES_DATA_PATH = "Coding Data 02 - standard/allergies.csv"
CAREPLANS_DATA_PATH = "Coding Data 02 - standard/careplans.csv"
CONDITIONS_DATA_PATH = "Coding Data 02 - standard/conditions_reduced.csv"
ENCOUNTER_DATA_PATH = "Coding Data 02 - standard/encounter_reduced.csv"
MEDICATIONS_DATA_PATH = "Coding Data 02 - standard/medications_reduced.csv"
OBSERVATIONS_DATA_PATH = "Coding Data 02 - standard/observations_reduced.csv"
PATIENTS_DATA_PATH = "Coding Data 02 - standard/patients.csv"
# Innitialisiere Dataframes
allergies_df = pd.read_csv(ALLERGIES_DATA_PATH)
conditions_df = pd.read_csv(CONDITIONS_DATA_PATH)
encounters_df = pd.read_csv(ENCOUNTER_DATA_PATH)
medications_df = pd.read_csv(MEDICATIONS_DATA_PATH)
observations_df = pd.read_csv(OBSERVATIONS_DATA_PATH)
patients_df = pd.read_csv(PATIENTS_DATA_PATH)
# Liste aller Dataframes mit zugehörigem Namen
df_list = [(allergies_df, "allergies"), (conditions_df, "conditions"), (encounters_df, "encounters"),
           (medications_df, "medications"), (observations_df, "observations"), (patients_df, "patients")]
# Printe jedes DF mit namen+ columns+ rows
for df in df_list:
    print(f"file:{df[1]}, variables(columns): {df[0].shape[1]}, entries(rows): {df[0].shape[0]}")



1.2

In [ ]:
# Daten zu Datetime Daten
patients_df["BIRTHDATE"] = pd.to_datetime(patients_df["BIRTHDATE"])
conditions_df["START"] = pd.to_datetime(conditions_df["START"])
#Aktuelles Alter bestimmen und in neue Column einfügen
patients_df["CURRENT_AGE"] = (pd.Timestamp.today() - patients_df["BIRTHDATE"]).dt.days // 365

# Vereinfacht spätere Berechnung
conditions = conditions_df.merge(
    patients_df[["Id", "BIRTHDATE"]],
    left_on="PATIENT",
    right_on="Id",
    how="left"
)
# Alter zum Start der Condition berechnen
conditions["AGE_AT_CONDITION_START"] = (conditions["START"] - conditions["BIRTHDATE"]).dt.days // 365

print("Average patient age:", patients_df["CURRENT_AGE"].mean())
print("Average age at condition start:", conditions["AGE_AT_CONDITION_START"].mean())
print(patients_df.head())


1.3

In [ ]:
#Funktion um Childhood Asthma Kinder zu filtern
def filter_children(patients):
    print("Records before filtering:", len(patients))

    filtered_patients = patients[
        (patients_df["CURRENT_AGE"] >= 6) &
        (patients_df["CURRENT_AGE"] <= 12)
        ]

    print("Records after filtering:", len(filtered_patients))

    return filtered_patients


children_patients = filter_children(patients_df)

1.4

In [ ]:
#Funktion um Childhood Asthma Kinder zu filtern
def filter_to_childhood_asthma_children(df, asthma_child_ids):
    before = len(df)

    if "PATIENT" in df.columns:
        filtered_df = df[df["PATIENT"].isin(asthma_child_ids)].copy()
    else:
        filtered_df = df[df["Id"].isin(asthma_child_ids)].copy()

    after = len(filtered_df)

    print("Records before filtering:", before)
    print("Records after filtering:", after)

    return filtered_df


# Children with Childhood asthma diagnosis finden
child_ids = children_patients["Id"]

childhood_asthma_conditions = conditions[
    (conditions["PATIENT"].isin(child_ids)) &
    (conditions["CODE"] == 233678006)
    ]
# alle unique asthma childs
asthma_child_ids = childhood_asthma_conditions["PATIENT"].unique()

print("Number of children with Childhood asthma:", len(asthma_child_ids))
# Filter jeweils nur Childhood Asthma Kinder aus DF's
patients_asthma_children = filter_to_childhood_asthma_children(
    children_patients,
    asthma_child_ids
)

conditions_asthma_children = filter_to_childhood_asthma_children(
    conditions,
    asthma_child_ids
)

encounters_asthma_children = filter_to_childhood_asthma_children(
    encounters_df,
    asthma_child_ids
)

medications_asthma_children = filter_to_childhood_asthma_children(
    medications_df,
    asthma_child_ids)

condition_counts = conditions_asthma_children.groupby("PATIENT").size()

children_more_than_one_condition = (condition_counts > 1).sum()
max_conditions = condition_counts.max()

print("Children with more than one condition:", children_more_than_one_condition)
print("Maximum number of conditions associated with one patient:", max_conditions)

2.1

In [ ]:
# Anzahl aller Conditions pro Kind
condition_count = conditions_asthma_children.groupby("PATIENT").size()

# Anzahl distinct Conditions pro Kind
unique_condition_count = conditions_asthma_children.groupby("PATIENT")["CODE"].nunique()

# Anzahl aller Encounters pro Kind
encounter_count = encounters_asthma_children.groupby("PATIENT").size()

# Anzahl aller Meds pro Kind
medication_count = medications_asthma_children.groupby("PATIENT").size()

# Zum Patient DataFrame hinzufügen
patients_asthma_children["CONDITION_COUNT"] = patients_asthma_children["Id"].map(condition_count)
patients_asthma_children["UNIQUE_CONDITION_COUNT"] = patients_asthma_children["Id"].map(unique_condition_count)
patients_asthma_children["ENCOUNTER_COUNT"] = patients_asthma_children["Id"].map(encounter_count)
patients_asthma_children["MEDICATION_COUNT"] = patients_asthma_children["Id"].map(medication_count)


# Einfacher Health Score
# Je höher der Wert desto schlimmer ist das Asthma
patients_asthma_children["HEALTH_SCORE"] = (
        patients_asthma_children["CONDITION_COUNT"] +
        patients_asthma_children["UNIQUE_CONDITION_COUNT"] +
        patients_asthma_children["ENCOUNTER_COUNT"] +
        patients_asthma_children["MEDICATION_COUNT"]
)

patients_asthma_children[[
    "Id",
    "CONDITION_COUNT",
    "UNIQUE_CONDITION_COUNT",
    "ENCOUNTER_COUNT",
    "MEDICATION_COUNT",
    "HEALTH_SCORE"
]].head()

3.1

In [ ]:

output_dir = Path.home() / "Documents" / "Uni" / "Uni" / "CRI" / "Exercise5"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "asthma_patient_summary.csv"

patients_asthma_children.to_csv(output_path, index=False)

print(f"CSV file exported successfully: {output_path}")